# XGBoost classifier 
using native categorical support(category dtype + enable_categorical=True, tree_method="hist").

In [2]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

fe = importlib.import_module("06_Feature_Engineering")
cv = importlib.import_module("07_Cross_Validation")

MODEL_NAME = "xgboost"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

XGB_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=5,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    early_stopping_rounds=150,
    random_state=cv.SEED,
    n_jobs=-1,
)

In [3]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    train, test = fe.load_raw_data()
    train = fe.engineer_features(train)
    test = fe.engineer_features(test)
    num_cols, cat_cols = fe.get_feature_lists(train)
    feature_cols = num_cols + cat_cols

    # XGBoost's native categorical support needs pandas "category" dtype.
    for c in cat_cols:
        train[c] = train[c].astype("category")
        test[c] = test[c].astype("category")

    y = train[fe.TARGET].values
    fold_ids = cv.get_or_create_folds(train, target_col=fe.TARGET)

    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
    importances = np.zeros(len(feature_cols))

    print("=" * 70)
    print(f"XGBOOST ({cv.N_SPLITS}-fold CV)")
    print("=" * 70)

    for fold in range(cv.N_SPLITS):
        train_idx, valid_idx = cv.fold_split(train, fold_ids, fold)

        X_train, y_train = train.loc[train_idx, feature_cols], y[train_idx]
        X_valid, y_valid = train.loc[valid_idx, feature_cols], y[valid_idx]

        model = XGBClassifier(**XGB_PARAMS)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=200,
        )

        valid_pred = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred

        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.best_iteration})")

        test_pred += model.predict_proba(test[feature_cols])[:, 1] / cv.N_SPLITS
        importances += model.feature_importances_ / cv.N_SPLITS
        model.save_model(f"{MODEL_DIR}/xgboost_fold{fold}.json")

    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    cv.summarize_oof(y, oof_pred, MODEL_NAME)

    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nTop feature importances:")
    print(imp_df.head(15).to_string(index=False))

    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({fe.ID_COL: test[fe.ID_COL], fe.TARGET: test_pred}).to_csv(
        SUB_PATH, index=False
    )
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")


In [4]:
if __name__ == "__main__":
    main()

XGBOOST (5-fold CV)
[0]	validation_0-auc:0.92325
[200]	validation_0-auc:0.94007
[400]	validation_0-auc:0.94016
[470]	validation_0-auc:0.94006
Fold 0: AUC = 0.94019 (best_iter=320)
[0]	validation_0-auc:0.92499
[200]	validation_0-auc:0.94105
[400]	validation_0-auc:0.94095
[416]	validation_0-auc:0.94093
Fold 1: AUC = 0.94109 (best_iter=266)
[0]	validation_0-auc:0.92613
[200]	validation_0-auc:0.94228
[400]	validation_0-auc:0.94229
[439]	validation_0-auc:0.94223
Fold 2: AUC = 0.94242 (best_iter=289)
[0]	validation_0-auc:0.92495
[200]	validation_0-auc:0.94191
[387]	validation_0-auc:0.94187
Fold 3: AUC = 0.94197 (best_iter=237)
[0]	validation_0-auc:0.92429
[200]	validation_0-auc:0.94131
[390]	validation_0-auc:0.94120
Fold 4: AUC = 0.94137 (best_iter=240)

Mean fold AUC: 0.94141 (+/- 0.00076)
[xgboost] OOF ROC-AUC: 0.94140

Top feature importances:
                      feature  importance
            Subsidy_Available    0.443472
         Subsidy_x_EnvConcern    0.307524
                  Sub